# 01. 지도학습 첫걸음 — 회귀와 분류

**대응 강의:** [1강 학습의 종류](../../Lecture/03-머신러닝-기본개념/01-학습의-종류.md), [2강 문제 정의](../../Lecture/03-머신러닝-기본개념/02-문제-정의.md)

머신러닝의 첫 단추를 직접 코드로 끼워볼 거예요. 이번에 같이 해볼 건:
- 데이터가 **특성(X)** 과 **타깃(y)** 으로 나뉘는 모습을 눈으로 직접 보기
- scikit-learn이 어떤 모델이든 똑같이 쓰는 **`fit` → `predict`** 흐름을 처음 체험해 보기
- 숫자를 맞히는 **회귀**와 종류를 맞히는 **분류**가 어떻게 다른지 코드로 비교해 보기

> ▶️ 위쪽 메뉴에서 **런타임 → 모두 실행** 을 누르면 한 번에 쭉 돌아가요. 아니면 셀마다 `Shift+Enter`로 하나씩 실행해도 좋아요.

In [ ]:
# Colab에는 아래 라이브러리가 이미 깔려 있어요. (따로 설치 안 해도 됩니다)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer

print('준비 완료! numpy', np.__version__)

## Part A. 회귀(Regression) — "얼마나?"를 맞히기

먼저 당뇨병 진행도 데이터를 가지고 놀아볼게요. 환자의 여러 측정값으로 **1년 뒤 질병이 얼마나 진행될지(연속된 숫자)** 를 예측하는 문제예요. 맞혀야 할 답이 숫자라서, 이런 걸 **회귀**라고 불러요.

In [ ]:
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

print('전체 모양 (행=환자 한 명, 열=특성+타깃):', df.shape)
print('특성(feature) 이름:', list(diabetes.feature_names))
print('타깃(target): 질병이 얼마나 진행됐는지 (연속 숫자)')
df.head()

In [ ]:
# 모든 지도학습은 여기서 출발해요: 입력(X)과 정답(y)을 갈라놓기
X = diabetes.data      # 입력으로 쓸 여러 특성
y = diabetes.target    # 우리가 맞히고 싶은 정답

print('X(특성) 모양:', X.shape, '  <- 442명 x 특성 10개')
print('y(타깃) 모양:', y.shape, '  <- 442명 각자의 정답')
print('타깃 예시 5개:', y[:5].round(1).tolist())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1) 데이터를 훈련용/시험용으로 나눠요 ('왜 나누는지'는 4강에서 자세히 다뤄요)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2) 모델 만들기 → fit(학습) → predict(예측). 이게 scikit-learn의 기본 3박자예요
model = LinearRegression()
model.fit(X_train, y_train)          # 학습: 데이터를 보면서 내부 손잡이(파라미터)를 맞춰가요
pred = model.predict(X_test)         # 예측: 한 번도 안 본 데이터에 적용해 봐요

# 3) 얼마나 잘 맞혔는지 채점 (회귀 지표는 로드맵 6번에서 더 깊게 봐요)
print('RMSE (낮을수록 좋아요):', mean_squared_error(y_test, pred) ** 0.5)
print('R^2  (1에 가까울수록 좋아요):', round(r2_score(y_test, pred), 3))

In [ ]:
# 예측이 얼마나 정확한지 그림으로 봐요: 점이 대각선에 붙어 있을수록 잘 맞힌 거예요
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, alpha=0.6)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', label='Perfect prediction (y=x)')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Regression: Predicted vs Actual')
plt.legend(); plt.show()

## Part B. 분류(Classification) — "어느 쪽?"을 맞히기

이번엔 유방암 데이터예요. 종양을 측정한 값들로 **양성인지 악성인지(둘 중 하나)** 를 판정하는 문제죠. 맞혀야 할 답이 숫자가 아니라 '종류'라서, 이런 건 **분류**라고 불러요.

In [ ]:
cancer = load_breast_cancer()
Xc, yc = cancer.data, cancer.target

print('특성 개수:', Xc.shape[1])
print('타깃 클래스:', dict(zip(cancer.target_names, [0, 1])), '  <- 숫자가 아니라 종류예요!')
print('클래스 분포:', np.bincount(yc), '(악성 개수, 양성 개수)')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

# 스케일링은 '훈련 데이터로만 fit' 하는 게 중요해요 — 안 그러면 데이터 누수예요 (4강 핵심!)
scaler = StandardScaler()
Xc_tr_s = scaler.fit_transform(Xc_tr)   # 훈련 데이터: fit + transform
Xc_te_s = scaler.transform(Xc_te)       # 시험 데이터: transform만 (훈련 때 기준을 그대로 적용)

clf = LogisticRegression(max_iter=5000)
clf.fit(Xc_tr_s, yc_tr)
pred_c = clf.predict(Xc_te_s)

print('정확도(accuracy):', round(accuracy_score(yc_te, pred_c), 3))

In [ ]:
# 분류 모델은 속으로 '확률'을 먼저 계산하고, 그걸 종류로 바꿔요 (2강 참고)
proba = clf.predict_proba(Xc_te_s)[:5]
for i, (p, label) in enumerate(zip(proba, pred_c[:5])):
    print(f'샘플{i}: 악성확률={p[0]:.2f}, 양성확률={p[1]:.2f} -> 최종판정={cancer.target_names[label]}')

# 혼동행렬 (로드맵 6번에서 자세히 다뤄요)
print('\n혼동행렬 [[TN, FP], [FN, TP]]:')
print(confusion_matrix(yc_te, pred_c))

## 정리

| | 회귀(Part A) | 분류(Part B) |
|---|---|---|
| 타깃 | 연속 숫자 | 종류(범주) |
| 모델 | `LinearRegression` | `LogisticRegression` |
| 평가 | RMSE, R² | 정확도, 혼동행렬 |
| 공통 | **`fit` → `predict`** 흐름은 완전히 똑같아요! | |

> 💡 scikit-learn 모델은 종류가 뭐든 `fit`/`predict` 쓰는 법이 같아요. 그래서 알고리즘만 바꿔 끼우면 되는 거예요.

## 🎯 직접 해보기

1. Part B에서 `LogisticRegression`을 `from sklearn.tree import DecisionTreeClassifier`의 `DecisionTreeClassifier()`로 바꿔 보세요. 정확도가 달라지나요? (쓰는 법은 그대로예요!)
2. Part A에서 `test_size=0.2`를 `0.5`로 바꾸면 R²가 어떻게 변하나요? 왜 그럴까요?
3. `load_iris`(붓꽃 3종 분류)를 불러와서 Part B처럼 분류해 보세요. 이건 종류가 3개라 **다중 분류**라고 불러요.